## Load Pretrained Model

A **pretrained model** is a CNN that has already been trained on a large image dataset.

Instead of training the entire CNN from the beginning, we reuse the knowledge learned by the pretrained model.

For this notebook, we can use a model available in **TorchVision**, such as **ResNet18**.


### What this code does

- `models.resnet18()` → loads the ResNet18 architecture.
- `weights="DEFAULT"` → loads pretrained weights.
- The model already contains learned features from its original training.

So instead of:

**Random weights → Train everything**

we start with:

**Pretrained weights → Adapt the model to our dataset**

### Important Point

The pretrained model is **not yet trained for our specific classification task**. We will modify its classification head and train it for our classes.

In [1]:

### Python Implementation

import torch
from torchvision import models

model = models.resnet18(weights="DEFAULT")

print(model)


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_sta

## Inspect Architecture

After loading the pretrained model, we should **inspect its architecture** to understand its main parts.

A CNN such as ResNet18 contains:

**Input → Feature Extraction Layers → Classification Head → Output**

The **feature extraction layers** learn useful image features such as edges, shapes, and patterns.

The **classification head** uses those learned features to predict the final classes.


### What this tells us

- `in_features=512` → the feature extractor produces 512 features.
- `out_features=1000` → the original model predicts 1000 classes.
- `fc` → this is the final classification head.

For our own dataset, we may have only **2 classes**, such as:

**Cat / Dog**

So we will replace the original classification head with a new one that produces 2 outputs.

### Main Idea

We don't need to understand every layer of ResNet18.

For transfer learning, the important idea is:

**Pretrained CNN → Feature Extractor + Classification Head**

We will keep the useful pretrained feature extractor and modify the classification head for our own task.

In [3]:

### Python Implementation

print(model)


#You can also inspect the final classification layer:

print(model.fc)


#For ResNet18, the output will be similar to:

Linear(in_features=512, out_features=1000, bias=True)


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_sta

NameError: name 'Linear' is not defined

## Freeze Backbone

The **backbone** is the main feature-extraction part of the pretrained CNN.

When we **freeze the backbone**, we tell PyTorch:

> “Do not change the pretrained weights of these layers during training.”

The pretrained model has already learned useful features such as **edges, shapes, textures, and patterns**.

So, initially, we keep those learned features unchanged and train only the new classification head.

### Why Freeze the Backbone?

Suppose the pretrained model has learned:

**Image → Edges → Shapes → Patterns → Useful features**

These features can often be reused for a new image-classification task.

Instead of retraining all those layers, we initially train only the new classifier.

### Main Idea

**Pretrained backbone → Frozen**

**New classification head → Trainable**

This is called **feature extraction**.

In the next step, we will replace the original classification head with one suitable for our own classes.

In [4]:
### Python Implementation

for param in model.parameters():
    param.requires_grad = False

#This freezes all parameters in the model.

#We can check whether a parameter is frozen:

#```python
for name, param in model.named_parameters():
    print(name, param.requires_grad)


conv1.weight False
bn1.weight False
bn1.bias False
layer1.0.conv1.weight False
layer1.0.bn1.weight False
layer1.0.bn1.bias False
layer1.0.conv2.weight False
layer1.0.bn2.weight False
layer1.0.bn2.bias False
layer1.1.conv1.weight False
layer1.1.bn1.weight False
layer1.1.bn1.bias False
layer1.1.conv2.weight False
layer1.1.bn2.weight False
layer1.1.bn2.bias False
layer2.0.conv1.weight False
layer2.0.bn1.weight False
layer2.0.bn1.bias False
layer2.0.conv2.weight False
layer2.0.bn2.weight False
layer2.0.bn2.bias False
layer2.0.downsample.0.weight False
layer2.0.downsample.1.weight False
layer2.0.downsample.1.bias False
layer2.1.conv1.weight False
layer2.1.bn1.weight False
layer2.1.bn1.bias False
layer2.1.conv2.weight False
layer2.1.bn2.weight False
layer2.1.bn2.bias False
layer3.0.conv1.weight False
layer3.0.bn1.weight False
layer3.0.bn1.bias False
layer3.0.conv2.weight False
layer3.0.bn2.weight False
layer3.0.bn2.bias False
layer3.0.downsample.0.weight False
layer3.0.downsample.1.weight Fa

You will see:

```text
False
False
False
...
```

`False` means that parameter will **not be updated during training**.

## Replace Classification Head

The pretrained ResNet18 was originally trained to classify **1000 classes**.

Our own dataset may have a different number of classes.

For example:

**Cat vs Dog → 2 classes**

So, we replace the original classification head with a new one.

For ResNet18, the classification head is:

```python
model.fc
```


### What happens here?

Before:

```text
ResNet18
   ↓
Feature extraction
   ↓
512 features
   ↓
1000-class classifier
```

After replacing the head:

```text
ResNet18
   ↓
Feature extraction
   ↓
512 features
   ↓
2-class classifier
```

The **backbone remains pretrained**, while the new classification head starts with new weights.

### Important Point

We previously froze the pretrained model:

```python
for param in model.parameters():
    param.requires_grad = False
```

When we replace `model.fc`, the new `Linear` layer has newly created parameters, which are **trainable by default**.

So now:

**Backbone → Frozen**

**New classification head → Trainable**

This allows us to train the classifier without changing the pretrained feature extractor.

In [5]:

### Python Implementation

import torch.nn as nn

num_classes = 2

model.fc = nn.Linear(512, num_classes)


## Train Classifier

Now that the pretrained **backbone is frozen** and the new **classification head is added**, we train the classifier using our own dataset.

Only the new classification head needs to learn our specific classes.

### What happens during training?

For each batch:

**Images → Frozen Backbone → Features → New Classifier → Prediction**

Then:

**Prediction + Actual Label → Loss → Update Classifier**

The important point is that the **backbone weights are not updated**.

Only:

```python
model.fc.parameters()
```

are passed to the optimizer, so the new classification head learns the new task.

### Why is this useful?

Because the pretrained backbone already contains useful visual knowledge.

We only need to teach the final classifier:

> “These extracted features belong to class 0, and these belong to class 1.”

This is usually much faster than training the entire CNN from scratch.

In [9]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

train_dataset = datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

print("Training samples:", len(train_dataset))

 19%|█▉        | 33.1M/170M [07:07<29:33, 77.5kB/s]  


KeyboardInterrupt: 

## Evaluate

After training the new classification head, we need to check how well the model performs on **unseen test images**.

We should not use the training data for evaluation because the model has already learned from it.

For CIFAR-10, we can use the separate **test dataset**.

### What is happening?

The test images go through:

**Test Image → Frozen Backbone → Classifier → Prediction**

Then we compare:

**Predicted class vs Actual class**

For example:

```text
Actual:    Cat
Predicted: Cat
→ Correct
```

```text
Actual:    Dog
Predicted: Cat
→ Wrong
```

The final accuracy tells us how well the transfer-learning model performs on **new images it did not train on**.

### Important

We use:

```python
model.eval()
```

because the model is now being evaluated, not trained.

And:

```python
with torch.no_grad():
```

because we don't need to calculate gradients during evaluation.

In [ ]:

### Python Implementation

test_dataset = datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

model.eval()

correct = 0
total = 0

with torch.no_grad():

    for images, labels in test_loader:

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total

print(f"Test Accuracy: {accuracy:.2f}%")


## Unfreeze Selected Layers

After training only the classification head, we can allow some of the pretrained layers to learn from our dataset.

This is called **unfreezing selected layers**.

We don't have to unfreeze the entire backbone. We can choose only the later layers because they contain more task-specific features.

For ResNet18, we can unfreeze the last residual block:

```text
Earlier layers → Frozen
Later layers   → Trainable
Classification head → Trainable
```

### Why Unfreeze Selected Layers?

The earlier layers usually learn general features such as:

**Edges → Textures → Basic shapes**

These features are useful across many image datasets, so we normally keep them frozen.

Later layers learn more specific patterns, so allowing them to adapt can improve performance on our new dataset.

### Main Idea

**Feature extraction:**

```text
Backbone → Frozen
Classifier → Trainable
```

**After unfreezing:**

```text
Earlier backbone layers → Frozen
Later backbone layers → Trainable
Classifier → Trainable
```

This prepares the model for the next step: **fine-tuning**.

In [ ]:

### Python Implementation

for param in model.layer4.parameters():
    param.requires_grad = True
#```

#Now `layer4` can be updated during training.

#We can check:

#```python id="5jv6nq"
for name, param in model.named_parameters():
    if param.requires_grad:
        print(name)
#```

#You should see parameters from:

#```text
layer4
fc
#```


## Fine-Tune

**Fine-tuning** means continuing to train a pretrained model after unfreezing some of its layers.

In the previous step, we:

- Kept the earlier layers frozen.
- Unfroze `layer4`.
- Kept the new classification head trainable.

Now we train these trainable layers using our dataset.

Because the pretrained layers already contain useful knowledge, we usually use a **smaller learning rate** during fine-tuning.


### What is different now?

Previously:

```text
Only classification head
        ↓
Training
```

Now:

```text
Selected backbone layers
        +
Classification head
        ↓
Fine-tuning
```

The learning rate is reduced:

```python
lr=0.0001
```

because we don't want to make large changes to the useful pretrained weights.

### Main Idea

**Fine-tuning = carefully adapting pretrained layers to our new dataset.**

It can improve the model when the new dataset is sufficiently related to the type of images used to pretrain the model.

In [ ]:

### Python Implementation

optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=0.0001
)

model.train()

for epoch in range(3):
    for images, labels in train_loader:

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

    print(f"Epoch {epoch + 1}, Loss: {loss.item():.4f}")


## Compare Results

Now we compare the model **before and after fine-tuning**.

We have two transfer-learning stages:

1. **Feature Extraction** — backbone frozen, only classifier trained.
2. **Fine-Tuning** — selected backbone layers + classifier trained.

Record the test accuracy from each stage.

### Python Implementation

```python
feature_extraction_accuracy = 0.0   # replace with your actual accuracy
fine_tuning_accuracy = 0.0          # replace with your actual accuracy

print("Feature Extraction Accuracy:", feature_extraction_accuracy)
print("Fine-Tuning Accuracy:", fine_tuning_accuracy)
```

For example, your results table can look like:

| Method | Test Accuracy |
|---|---:|
| Feature Extraction | 80% |
| Fine-Tuning | 84% |

The actual values will depend on your training.

### What should we observe?

If fine-tuning gives higher accuracy:

**Fine-tuning improved the model's adaptation to the new dataset.**

If the improvement is very small, the pretrained features may already have been sufficient.

If fine-tuning performs worse, the model may have been changed too much or the learning rate may be too high.

### Important

Do not assume that fine-tuning will **always** produce better results.

The purpose of this experiment is to **measure and compare the results**.

## Training From Scratch vs Transfer Learning

There are two common ways to train a CNN.

### Training From Scratch

The model starts with **randomly initialized weights**.

```text
Random weights
      ↓
Train entire CNN
      ↓
Learn features
      ↓
Learn classification
```

The model must learn everything from the available dataset.

### Transfer Learning

The model starts with **pretrained weights**.

```text
Pretrained weights
      ↓
Reuse learned features
      ↓
Train new classifier
      ↓
Fine-tune selected layers
```

The model already has useful visual knowledge.

### Comparison

| Factor | Training From Scratch | Transfer Learning |
|---|---|---|
| **Training time** | Usually longer | Usually shorter |
| **Data requirement** | Usually needs more data | Often works well with less data |
| **Performance** | Can be good with enough data and training | Often gives strong performance quickly |
| **Computational cost** | Usually higher | Usually lower, especially when backbone is frozen |
| **Generalization** | Depends heavily on dataset size and training | Often benefits from pretrained general visual features |

### 1. Training Time

Training from scratch requires the CNN to learn features from the beginning.

Transfer learning starts with learned features, so training can be much faster, particularly when only the classification head is trained.

### 2. Data Requirement

Training a CNN from scratch generally requires a larger dataset.

Transfer learning can be useful when the new dataset is relatively small because the pretrained model already contains useful visual features.

### 3. Performance

Transfer learning often provides good performance without requiring extensive training.

Training from scratch can eventually perform very well when there is enough relevant data and computational resources.

### 4. Computational Cost

Training from scratch updates the entire network, so it generally requires more computation.

With transfer learning, freezing the backbone means only a small portion of the model is updated.

### 5. Generalization

A pretrained model has learned general visual patterns from a large dataset.

These patterns can help the model generalize to new images.

However, transfer learning is not automatically better. If the new dataset is very different from the pretrained model's original domain, its learned features may be less useful.

### Final Understanding

**Training from scratch:**

> Learn everything from the beginning.

**Transfer learning:**

> Reuse existing knowledge and adapt it to a new task.

For many image-classification problems, transfer learning is a practical starting point because it can reduce **training time, data requirements, and computational cost** while providing strong performance.